# 🧬 GEMA — Modelo predictivo del gemelo digital metabólico

**Proyecto:** GEMA · Gemelo digital metabólico (Lima, Perú) · MVP para feria de proyectos

Este notebook desarrolla los **dos modelos predictivos** del gemelo digital, siguiendo la guía técnica del proyecto:

| Modelo | Algoritmo (MVP) | Algoritmo (producción) | Predice | Meta |
|---|---|---|---|---|
| 1. Clasificador de riesgo | Random Forest (200 árboles) | Random Forest | riesgo {bajo, moderado, alto} | accuracy > 85 % |
| 2. Predictor de pico glucémico | Gradient Boosting | LSTM (con serie CGM 12 h) | glucosa a 60 min post-comida (mg/dL) | MAE < 15 mg/dL |

### Referencias que sustentan el diseño
- **Zeevi et al. 2015 (Cell)** — *Personalized Nutrition by Prediction of Glycemic Responses*: demostró que la respuesta glucémica postprandial (PPGR) se puede predecir con **gradient boosting** a partir de la comida + características de la persona. Es el paper base de Twin Health y de nuestro Modelo 2.
- **Shamanna et al. 2020-2021 (Twin Health)** — gemelo digital de cuerpo completo: 71 % de remisión de diabetes tipo 2 en 1 año vs 2.4 % en atención estándar.
- **Reynolds et al. 2016 (Diabetologia)** — caminar después de comer reduce el pico postprandial hasta ~20 %.
- **Spiegel et al. 1999 (Lancet)** — dormir < 7 h eleva la resistencia a la insulina ~30 % al día siguiente.
- **Simental-Mendía et al. 2008** — índice TyG = ln(TG/2) + ln(glucosa ayunas/2) como proxy de resistencia a la insulina.
- **Gurka & DeBoer 2014** — score continuo de severidad del síndrome metabólico (MetS-Z), inspiración del ICM.

> ⚠️ **Datos:** mientras no haya usuarios reales con CGM, entrenamos con una **población sintética** cuyas distribuciones replican NHANES/ENDES y las relaciones fisiológicas publicadas. El pipeline queda listo para re-entrenar con datos reales sin cambiar una línea.


## 1 · Setup


In [ ]:
# En Colab solo necesitas esto (sklearn y pandas ya vienen preinstalados)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, mean_absolute_error, r2_score)
plt.style.use('dark_background')


## 2 · Generador de datos sintéticos

Cada fila del **dataset de riesgo** = resumen de 7 días de un usuario (lo que el backend agregaría desde CGM + wearable). Cada fila del **dataset de comidas** = un evento de comida con su pico observado a 60 min.

Las correlaciones están moldeadas según la literatura (poco sueño → más variabilidad glucémica; caminata post-comida → menor pico; etc.).


In [ ]:
"""Generador de datos sintéticos fisiológicamente plausibles para GEMA.

Mientras no tengamos datos reales de usuarios (CGM + wearable), entrenamos con
una población sintética cuyas distribuciones replican lo publicado:

  - NHANES (USA) y ENDES (Perú): distribuciones de cintura, glucosa en ayunas,
    triglicéridos en adultos 20-45 años.
  - Zeevi et al. 2015 (Cell): la respuesta glucémica postprandial (PPGR)
    depende de la comida (carbohidratos, carga glucémica) Y de la persona
    (glucosa basal, actividad, sueño) — el mismo plato produce picos distintos
    en personas distintas.
  - Reynolds et al. 2016 (Diabetologia): caminar después de comer reduce el
    pico postprandial hasta ~20 %.
  - Spiegel et al. 1999 (Lancet): dormir < 7 h aumenta la resistencia a la
    insulina al día siguiente (~30 %).
  - Simental-Mendía et al. 2008: índice TyG como proxy de resistencia a la
    insulina. TyG = ln(TG/2) + ln(glucosa_ayunas/2).

El generador introduce correlaciones realistas (poco sueño → más variabilidad
glucémica; pocos pasos → más cintura) y ruido individual, para que el modelo
aprenda relaciones reales y no artefactos.
"""

import numpy as np
import pandas as pd

RNG_SEED = 42


# ─────────────────────────────────────────────────────────────────────────────
# Dataset 1 — Clasificación de riesgo metabólico (bajo / moderado / alto)
# ─────────────────────────────────────────────────────────────────────────────

def generar_poblacion_riesgo(n: int = 6000, seed: int = RNG_SEED) -> pd.DataFrame:
    """Población sintética de adultos 20-45 años de Lima Metropolitana.

    Cada fila = el resumen de 7 días de un usuario (lo que el Random Forest
    consumirá en producción según la guía técnica del proyecto).
    """
    rng = np.random.default_rng(seed)

    # Factor latente de "carga metabólica" por persona (0 = óptimo, 1 = muy mal)
    latente = rng.beta(2.2, 2.8, n)

    # Hábitos — correlacionados con el factor latente + ruido individual
    sueno_h = np.clip(8.2 - 2.8 * latente + rng.normal(0, 0.7, n), 3.5, 10)
    pasos = np.clip(11000 - 7500 * latente + rng.normal(0, 1500, n), 800, 20000)
    hrv_ms = np.clip(72 - 38 * latente + rng.normal(0, 8, n), 12, 110)

    # Antropometría / laboratorio
    cintura_cm = np.clip(74 + 30 * latente + rng.normal(0, 5, n), 60, 130)
    glucosa_ayunas = np.clip(82 + 38 * latente + rng.normal(0, 6, n), 65, 160)
    trigliceridos = np.clip(90 + 130 * latente + rng.normal(0, 25, n), 40, 400)
    tyg = np.log(trigliceridos / 2) + np.log(glucosa_ayunas / 2)

    # Indicadores CGM de 7 días (derivados de hábitos + ruido)
    cv_glucemico = np.clip(
        12 + 18 * latente + (7 - np.minimum(sueno_h, 7)) * 1.5 + rng.normal(0, 2.5, n),
        6, 48,
    )
    tiempo_en_rango = np.clip(
        96 - 42 * latente - (cv_glucemico - 12) * 0.8 + rng.normal(0, 4, n),
        20, 100,
    )
    pico_postprandial = np.clip(
        118 + 62 * latente - (pasos / 1000) * 0.9 + rng.normal(0, 8, n),
        100, 240,
    )

    df = pd.DataFrame({
        "cv_glucemico_pct": cv_glucemico.round(1),
        "tiempo_en_rango_pct": tiempo_en_rango.round(1),
        "pico_postprandial_prom": pico_postprandial.round(0),
        "sueno_promedio_h": sueno_h.round(1),
        "pasos_promedio_dia": pasos.round(0),
        "hrv_promedio_ms": hrv_ms.round(0),
        "indice_tyg": tyg.round(2),
        "cintura_cm": cintura_cm.round(0),
    })

    # Etiqueta: score continuo tipo MetS-Z simplificado → 3 clases
    score = (
        0.25 * (df.cv_glucemico_pct - 12) / 18
        + 0.20 * (100 - df.tiempo_en_rango_pct) / 50
        + 0.15 * (df.pico_postprandial_prom - 118) / 60
        + 0.10 * (7.5 - df.sueno_promedio_h).clip(0) / 3
        + 0.10 * (9000 - df.pasos_promedio_dia).clip(0) / 7000
        + 0.10 * (df.indice_tyg - 8.0) / 1.2
        + 0.10 * (df.cintura_cm - 80) / 30
        + np.random.default_rng(seed + 1).normal(0, 0.05, n)  # ruido de etiqueta
    )
    df["riesgo"] = pd.cut(
        score, bins=[-np.inf, 0.28, 0.55, np.inf],
        labels=["bajo", "moderado", "alto"],
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Dataset 2 — Predicción del pico glucémico postprandial (regresión)
# ─────────────────────────────────────────────────────────────────────────────

def generar_comidas(n: int = 9000, seed: int = RNG_SEED) -> pd.DataFrame:
    """Eventos de comida con el pico de glucosa observado a 60 min.

    En producción este modelo será un LSTM con la serie CGM de 12 h
    (según la guía técnica); este dataset entrena el baseline tabular
    (Gradient Boosting), el mismo enfoque del paper de Zeevi 2015.
    """
    rng = np.random.default_rng(seed)

    glucosa_basal = np.clip(rng.normal(95, 12, n), 70, 150)
    carbos_g = np.clip(rng.gamma(4.5, 14, n), 5, 220)            # típico 30-90 g
    indice_glucemico = np.clip(rng.normal(62, 16, n), 25, 105)   # IG del plato
    caminata_post_min = np.clip(rng.exponential(12, n), 0, 60)
    sueno_anoche_h = np.clip(rng.normal(6.6, 1.2, n), 3.5, 10)
    hrv_ms = np.clip(rng.normal(48, 16, n), 12, 110)
    hora_dia = rng.integers(6, 23, n)

    # Carga glucémica = carbos * IG / 100 (definición estándar)
    carga_glucemica = carbos_g * indice_glucemico / 100

    # Modelo generador del pico (fisiología publicada + interacciones):
    deficit_sueno = np.clip(7.0 - sueno_anoche_h, 0, None)
    estres = np.clip((55 - hrv_ms) / 40, 0, None)

    pico_60min = (
        glucosa_basal
        + 0.95 * carga_glucemica                       # efecto principal comida
        + 0.10 * carbos_g                              # carbos absolutos
        - 0.55 * caminata_post_min                     # Reynolds 2016
        * (1 + 0.3 * (carga_glucemica > 35))           #   camina más → más efecto si comida pesada
        + 6.5 * deficit_sueno                          # Spiegel 1999
        + 9.0 * estres                                 # cortisol ↑ glucosa
        + 4.0 * ((hora_dia >= 20) | (hora_dia <= 7))   # peor tolerancia nocturna
        + rng.normal(0, 7.5, n)                        # variabilidad individual
    )
    pico_60min = np.clip(pico_60min, glucosa_basal - 5, 280)

    return pd.DataFrame({
        "glucosa_basal_mgdl": glucosa_basal.round(0),
        "carbohidratos_g": carbos_g.round(0),
        "indice_glucemico": indice_glucemico.round(0),
        "carga_glucemica": carga_glucemica.round(1),
        "caminata_post_min": caminata_post_min.round(0),
        "sueno_anoche_h": sueno_anoche_h.round(1),
        "hrv_ms": hrv_ms.round(0),
        "hora_dia": hora_dia,
        "pico_60min_mgdl": pico_60min.round(0),
    })


if __name__ == "__main__":
    r = generar_poblacion_riesgo(8)
    c = generar_comidas(8)
    print(r.to_string(index=False))
    print()
    print(c.to_string(index=False))


## 3 · Modelo 1 — Clasificador de riesgo metabólico (Random Forest)

**Input (8 features del resumen semanal):** CV glucémico, tiempo en rango 70-140, pico postprandial promedio, sueño promedio, pasos/día, HRV, índice TyG, cintura.

**Output:** clase de riesgo `bajo / moderado / alto` + probabilidades (la app las usa para el color del gemelo y las alertas).


In [ ]:
FEATURES_RIESGO = ['cv_glucemico_pct','tiempo_en_rango_pct','pico_postprandial_prom',
                   'sueno_promedio_h','pasos_promedio_dia','hrv_promedio_ms',
                   'indice_tyg','cintura_cm']

df_r = generar_poblacion_riesgo(6000)
Xr, yr = df_r[FEATURES_RIESGO], df_r['riesgo']
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, stratify=yr, random_state=7)

rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5,
                            class_weight='balanced', random_state=7, n_jobs=-1)
rf.fit(Xr_tr, yr_tr)
pred = rf.predict(Xr_te)
print(f'Accuracy test: {accuracy_score(yr_te, pred):.3f}  (meta > 0.85)')
print(classification_report(yr_te, pred))


In [ ]:
# Validación cruzada 5-fold + matriz de confusión
cv5 = cross_val_score(rf, Xr, yr, cv=5, scoring='accuracy')
print(f'CV 5-fold: {cv5.mean():.3f} ± {cv5.std():.3f}')
cm = confusion_matrix(yr_te, pred, labels=['bajo','moderado','alto'])
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3), ['bajo','moderado','alto']); ax.set_yticks(range(3), ['bajo','moderado','alto'])
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión — riesgo')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ¿Qué variables pesan más en el riesgo? (explicabilidad para el médico)
imp = pd.Series(rf.feature_importances_, index=FEATURES_RIESGO).sort_values()
imp.plot.barh(figsize=(7,4), color='#4DA3FF', title='Importancia de variables — Random Forest')
plt.tight_layout(); plt.show()


## 4 · Modelo 2 — Predictor del pico glucémico postprandial

**Input (evento de comida):** glucosa basal, carbohidratos (g), índice glucémico del plato, carga glucémica, minutos de caminata después, horas de sueño anoche, HRV, hora del día.

**Output:** pico de glucosa estimado a 60 min (mg/dL). Es el número que la app muestra al registrar una comida (\"este plato podría subir tu glucosa a ~158 mg/dL\").

En el MVP usamos **Gradient Boosting** (igual que Zeevi 2015). En producción se reemplaza por el **LSTM** de la guía técnica cuando existan series CGM de 12 h por usuario.


In [ ]:
FEATURES_PICO = ['glucosa_basal_mgdl','carbohidratos_g','indice_glucemico','carga_glucemica',
                 'caminata_post_min','sueno_anoche_h','hrv_ms','hora_dia']

df_c = generar_comidas(9000)
Xc, yc = df_c[FEATURES_PICO], df_c['pico_60min_mgdl']
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=7)

gbm = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                                subsample=0.85, random_state=7)
gbm.fit(Xc_tr, yc_tr)
pred_c = gbm.predict(Xc_te)
print(f'MAE: {mean_absolute_error(yc_te, pred_c):.1f} mg/dL  (meta < 15)')
print(f'R² : {r2_score(yc_te, pred_c):.3f}')


In [ ]:
# Predicho vs real
fig, ax = plt.subplots(figsize=(5.5,5))
ax.scatter(yc_te, pred_c, s=6, alpha=0.35, color='#1FD0A3')
lims = [yc_te.min(), yc_te.max()]
ax.plot(lims, lims, '--', color='#FF5E6C', lw=1.5)
ax.set_xlabel('Pico real (mg/dL)'); ax.set_ylabel('Pico predicho (mg/dL)')
ax.set_title('Modelo 2 — predicho vs real'); plt.tight_layout(); plt.show()


## 5 · El modelo en acción — los mismos escenarios de la app

Estos son los escenarios que GEMA muestra: registrar un arroz con pollo y simular qué pasa si caminas o cambias el arroz por quinua.


In [ ]:
base = {'glucosa_basal_mgdl': 95, 'carbohidratos_g': 78, 'indice_glucemico': 72,
        'carga_glucemica': 78*72/100, 'caminata_post_min': 0,
        'sueno_anoche_h': 5.7, 'hrv_ms': 42, 'hora_dia': 13}
escenarios = {
    'Arroz con pollo, sin caminar': base,
    '+ caminar 25 min': {**base, 'caminata_post_min': 25},
    'con quinua (IG 53)': {**base, 'indice_glucemico': 53, 'carga_glucemica': 78*53/100},
    'quinua + caminar': {**base, 'indice_glucemico': 53, 'carga_glucemica': 78*53/100, 'caminata_post_min': 25},
}
for nombre, esc in escenarios.items():
    pico = gbm.predict(pd.DataFrame([esc]))[0]
    print(f'{nombre:32s} → {pico:5.0f} mg/dL', '⚠' if pico > 140 else '✓')


In [ ]:
# Clasificación del usuario de la app (Fransua, 22 años, Lima)
fransua = pd.DataFrame([{'cv_glucemico_pct': 24.5, 'tiempo_en_rango_pct': 68.0,
    'pico_postprandial_prom': 158.0, 'sueno_promedio_h': 6.1,
    'pasos_promedio_dia': 6420, 'hrv_promedio_ms': 42,
    'indice_tyg': 8.30, 'cintura_cm': 88}])
print('Riesgo:', rf.predict(fransua)[0])
print('Probabilidades:', dict(zip(rf.classes_, rf.predict_proba(fransua)[0].round(3))))


## 6 · Conexión con la app GEMA

| Pantalla de la app | Modelo que la alimenta |
|---|---|
| Pill de riesgo (\"Riesgo moderado\") + color del gemelo | Modelo 1 (clases + probabilidades) |
| \"Este plato podría subir tu glucosa a ~158 mg/dL\" | Modelo 2 |
| Simulador what-if (caminar / dormir / carbos) | Modelo 2 evaluado en escenarios contrafactuales |
| Recomendaciones (quinua, caminata) | Comparación de escenarios del Modelo 2 |
| Reporte médico (riesgo a 5 años) | Modelo 1 + proyección epidemiológica |

### Roadmap del modelo (de MVP a producción)
1. **Hoy (MVP):** modelos entrenados con datos sintéticos fisiológicos → demuestran el pipeline completo.
2. **Piloto:** recolectar 2 semanas de calibración por usuario (CGM + wearable) → fine-tuning individual.
3. **Producción:** reemplazar el GBM por el LSTM de la guía (serie CGM 12 h → predicción a 30/60/120 min), reentrenamiento semanal por usuario.
4. **Validación clínica:** comparar predicciones vs CGM real, meta MAE < 15 mg/dL en datos reales.
